In [0]:
%sql
SELECT 'customer_products' AS tabla,
       COUNT(*) AS filas,
       COUNT(DISTINCT pk_cid, pk_partition) AS pares_cliente_mes,
       COUNT(DISTINCT pk_cid) AS clientes,
       COUNT(DISTINCT pk_partition) AS meses,
       MIN(pk_partition) AS primer_mes,
       MAX(pk_partition) AS ultimo_mes
FROM workspace.easymoney_bronze.customer_products
UNION ALL
SELECT 'customer_commercial_activity', COUNT(*), COUNT(DISTINCT pk_cid, pk_partition),
       COUNT(DISTINCT pk_cid), COUNT(DISTINCT pk_partition), MIN(pk_partition), MAX(pk_partition)
FROM workspace.easymoney_bronze.customer_commercial_activity
UNION ALL
SELECT 'customer_sociodemographics', COUNT(*), COUNT(DISTINCT pk_cid, pk_partition),
       COUNT(DISTINCT pk_cid), COUNT(DISTINCT pk_partition), MIN(pk_partition), MAX(pk_partition)
FROM workspace.easymoney_bronze.customer_sociodemographics;

In [0]:
%sql
SELECT
  SUM(CASE WHEN pk_cid IS NULL THEN 1 ELSE 0 END) AS nulos_cid,
  SUM(CASE WHEN pk_partition NOT RLIKE '^[0-9]{4}-[0-9]{2}$' THEN 1 ELSE 0 END) AS meses_formato_raro,
  SUM(CASE WHEN payroll IS NULL THEN 1 ELSE 0 END) AS nulos_payroll,
  SUM(CASE WHEN pension_plan IS NULL THEN 1 ELSE 0 END) AS nulos_pension_plan
FROM workspace.easymoney_bronze.customer_products;

In [0]:
%sql
SELECT COUNT(*) FROM workspace.easymoney_bronze.customer_products
WHERE payroll IS NULL AND pension_plan IS NULL;  

In [0]:
from pyspark.sql import functions as F

BRONZE = "workspace.easymoney_bronze"
SILVER = "workspace.easymoney_silver"


PRODUCT_COLS = [
    "short_term_deposit", "loans", "mortgage", "funds", "securities",
    "long_term_deposit", "em_account_pp", "credit_card", "payroll",
    "pension_plan", "payroll_account", "emc_account", "debit_card",
    "em_account_p", "em_acount",
]
RENAMES = {"em_acount": "em_account"}  # errata en origen

products_silver = spark.table(f"{BRONZE}.customer_products").select(
    F.col("pk_cid").cast("int").alias("customer_id"),
    F.to_date("pk_partition", "yyyy-MM").alias("partition_month"),
    *[
        # "0.0" -> double -> int; los 61 nulos de payroll/pension_plan -> 0
        F.coalesce(F.col(c).cast("double").cast("int"), F.lit(0)).alias(RENAMES.get(c, c))
        for c in PRODUCT_COLS
    ],
)

In [0]:
final_cols = [RENAMES.get(c, c) for c in PRODUCT_COLS]

n_rows = products_silver.count()
n_keys = products_silver.select("customer_id", "partition_month").distinct().count()
n_bad_flags = products_silver.filter(
    " OR ".join(f"{c} NOT IN (0, 1)" for c in final_cols)
).count()

assert n_rows == n_keys, f"Hay {n_rows - n_keys} duplicados cliente-mes"
assert n_bad_flags == 0, f"Hay {n_bad_flags} filas con flags distintos de 0/1"
print(f"OK: {n_rows:,} filas, sin duplicados, flags binarios")

In [0]:
(products_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{SILVER}.customer_products"))

spark.sql(f"""
  COMMENT ON TABLE {SILVER}.customer_products IS
  'Productos contratados por cliente y mes. Grano: customer_id + partition_month.
   61 nulos en payroll y pension_plan rellenados con 0 (0,001% de filas).
   Columna em_acount renombrada a em_account.'
""")

In [0]:
from pyspark.sql import functions as F

for t in ["customer_sociodemographics", "customer_commercial_activity"]:
    df = (spark.table(f"workspace.easymoney_bronze.{t}")
          .drop("_c0", "_source_file", "_ingested_at"))
    print(f"===== {t} =====")
    display(df.limit(5))
    display(df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]))

In [0]:
from pyspark.sql import functions as F

BRONZE = "workspace.easymoney_bronze"
SILVER = "workspace.easymoney_silver"

def assert_unique_grain(df, keys=("customer_id", "partition_month")):
    n_rows = df.count()
    n_keys = df.select(*keys).distinct().count()
    assert n_rows == n_keys, f"{n_rows - n_keys:,} duplicados en {keys}"
    return n_rows

def write_silver(df, table, comment):
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", True)
       .saveAsTable(f"{SILVER}.{table}"))
    spark.sql(f"COMMENT ON TABLE {SILVER}.{table} IS '{comment}'")

In [0]:
socio_silver = spark.table(f"{BRONZE}.customer_sociodemographics").select(
    F.col("pk_cid").cast("int").alias("customer_id"),
    F.to_date("pk_partition", "yyyy-MM").alias("partition_month"),
    F.col("country_id"),
    F.col("region_code").cast("double").cast("int").alias("region_code"),  # "29.0" -> 29
    F.col("gender"),                                                       
    F.col("age").cast("int").alias("age"),
    F.when(F.col("deceased") == "S", True)
     .when(F.col("deceased") == "N", False).alias("is_deceased"),
    F.round(F.col("salary").cast("double"), 2).alias("salary"),           # nulos se mantienen
)

In [0]:
n = assert_unique_grain(socio_silver)

n_bad_deceased = socio_silver.filter(F.col("is_deceased").isNull()).count()
n_bad_gender = socio_silver.filter(
    F.col("gender").isNotNull() & ~F.col("gender").isin("H", "V")
).count()

assert n_bad_deceased == 0, f"{n_bad_deceased} valores de deceased fuera de S/N"
assert n_bad_gender == 0, f"{n_bad_gender} valores de gender fuera de H/V"
print(f"OK: {n:,} filas")

In [0]:
display(socio_silver.agg(
    F.min("age").alias("edad_min"),
    F.max("age").alias("edad_max"),
    F.round(F.avg(F.col("salary").isNull().cast("int")) * 100, 1).alias("pct_salary_nulo"),
))

# ¿Los nulos de region_code son clientes fuera de España?
display(socio_silver.groupBy(
    (F.col("country_id") == "ES").alias("es_espana"),
    F.col("region_code").isNull().alias("region_nula"),
).count())

In [0]:
write_silver(socio_silver, "customer_sociodemographics",
    "Datos sociodemograficos por cliente y mes. Grano: customer_id + partition_month. "
    "salary con ~26 por ciento de nulos, no imputados (se decide en gold). "
    "region_code solo aplica a clientes de ES. gender sin recodificar (H/V no definido en diccionario).")

In [0]:
activity_raw = spark.table(f"{BRONZE}.customer_commercial_activity")

activity_silver = activity_raw.select(
    F.col("pk_cid").cast("int").alias("customer_id"),
    F.to_date("pk_partition", "yyyy-MM").alias("partition_month"),
    F.expr("CAST(try_to_timestamp(entry_date, 'yyyy-MM') AS DATE)").alias("entry_month"),
    F.col("entry_channel"),
    F.col("active_customer").cast("double").cast("int").alias("is_active"),  # "1.0" -> 1
    F.expr("try_cast(split(segment, ' - ')[0] AS INT)").alias("segment_code"),  # "03 - UNIVERSITARIO" -> 3
    F.trim(F.split("segment", " - ").getItem(1)).alias("segment_name"),         # -> "UNIVERSITARIO"
)

In [0]:
n = assert_unique_grain(activity_silver)

n_bad_entry = activity_raw.filter(
    F.col("entry_date").isNotNull()
    & F.expr("try_to_timestamp(entry_date, 'yyyy-MM') IS NULL")
).count()
n_bad_segment = activity_raw.filter(
    F.col("segment").isNotNull()
    & F.expr("try_cast(split(segment, ' - ')[0] AS INT) IS NULL")
).count()
n_bad_active = activity_silver.filter(~F.col("is_active").isin(0, 1)).count()

assert n_bad_entry == 0, f"{n_bad_entry:,} entry_date con formato inesperado"
assert n_bad_segment == 0, f"{n_bad_segment:,} segment con formato inesperado"
assert n_bad_active == 0, f"{n_bad_active:,} active_customer fuera de 0/1"
print(f"OK: {n:,} filas")

In [0]:
# Catálogo de segmentos
display(activity_silver.groupBy("segment_code", "segment_name").count().orderBy("segment_code"))

# Incoherencia temporal: alta posterior a la foto del mes
print("entry_month > partition_month:",
      activity_silver.filter(F.col("entry_month") > F.col("partition_month")).count())

# ¿Los nulos de canal y segmento son los mismos clientes?
display(activity_silver.groupBy(
    F.col("entry_channel").isNull().alias("canal_nulo"),
    F.col("segment_code").isNull().alias("segmento_nulo"),
).count())

In [0]:
write_silver(activity_silver, "customer_commercial_activity",
    "Actividad comercial por cliente y mes. Grano: customer_id + partition_month. "
    "entry_date (yyyy-MM) convertido a entry_month. segment separado en segment_code y segment_name. "
    "Nulos de entry_channel y segment (~2 por ciento) se mantienen; se tratan en gold.")

In [0]:
   %sql
   SELECT * FROM workspace.easymoney_bronze.sales LIMIT 5;